# Day 064 — Exercise 2: ContentStore

Every AI app that generates content needs a way to store and retrieve past generations. A `ContentStore` is the in-memory version — fast, simple, and good enough for an MVP. (Day 54 taught SQLAlchemy for durable storage; that's the upgrade path once the product is validated.)

Each item has: `content_id` (random), `user_id`, `prompt`, `content`, `created_at` (ISO-8601 UTC).

In [ ]:
import secrets
from datetime import datetime

# secrets.token_urlsafe(8) generates an 11-char URL-safe random ID
# Use it for content_id generation


## Task

Implement `ContentStore` with four methods:

- `add(user_id, prompt, content) -> str` — store item, return `content_id`
- `get(content_id) -> dict | None` — retrieve by id
- `list_user(user_id) -> list[dict]` — all items for one user
- `count(user_id) -> int` — how many items a user has

Use `secrets.token_urlsafe(8)` for unique IDs.

## Your Implementation

In [ ]:
class ContentStore:
    """In-memory store for generated content items.

    Each item: {content_id, user_id, prompt, content, created_at}
    created_at: ISO-8601 UTC string  (datetime.utcnow().isoformat() + 'Z')
    """

    def __init__(self):
        self._store: dict = {}   # content_id -> item dict

    def add(self, user_id: str, prompt: str, content: str) -> str:
        """Store item and return its content_id."""
        # TODO: generate content_id, build item dict, store, return id
        raise NotImplementedError

    def get(self, content_id: str) -> dict | None:
        """Return item dict or None if not found."""
        # TODO
        raise NotImplementedError

    def list_user(self, user_id: str) -> list[dict]:
        """Return all items for the given user_id."""
        # TODO
        raise NotImplementedError

    def count(self, user_id: str) -> int:
        """Return number of items for the given user_id."""
        # TODO
        raise NotImplementedError


In [ ]:
class ContentStore:
    def __init__(self):
        self._store: dict = {}

    def add(self, user_id: str, prompt: str, content: str) -> str:
        cid = secrets.token_urlsafe(8)
        self._store[cid] = {
            "content_id": cid,
            "user_id":    user_id,
            "prompt":     prompt,
            "content":    content,
            "created_at": datetime.utcnow().isoformat() + "Z",
        }
        return cid

    def get(self, content_id: str) -> dict | None:
        return self._store.get(content_id)

    def list_user(self, user_id: str) -> list[dict]:
        return [v for v in self._store.values() if v["user_id"] == user_id]

    def count(self, user_id: str) -> int:
        return sum(1 for v in self._store.values() if v["user_id"] == user_id)


## Automated checks

In [ ]:
score, total = 0, 6
try:
    store = ContentStore()

    # add returns a content_id string
    cid = store.add("alice", "prompt1", "content1")
    assert isinstance(cid, str) and len(cid) > 0
    score += 1; print("\u2705 add returns a non-empty content_id string")

    # get returns the item
    item = store.get(cid)
    assert item is not None
    assert item["user_id"] == "alice"
    assert item["prompt"] == "prompt1"
    assert item["content"] == "content1"
    assert "content_id" in item and "created_at" in item
    score += 1; print("\u2705 get returns correct item with all fields")

    # get returns None for unknown id
    assert store.get("nonexistent") is None
    score += 1; print("\u2705 get returns None for unknown content_id")

    # add more items for two users
    store.add("alice", "p2", "c2")
    store.add("bob",   "p3", "c3")
    store.add("alice", "p4", "c4")

    # list_user returns only that user's items
    alice_items = store.list_user("alice")
    assert len(alice_items) == 3  # alice added 3 items
    assert all(i["user_id"] == "alice" for i in alice_items)
    score += 1; print("\u2705 list_user returns only the user's items")

    # count
    assert store.count("alice") == 3
    assert store.count("bob")   == 1
    assert store.count("carol") == 0
    score += 1; print("\u2705 count returns correct per-user count")

    # IDs are unique
    ids = [store.add("x", "p", "c") for _ in range(10)]
    assert len(set(ids)) == 10
    score += 1; print("\u2705 content_ids are unique across 10 additions")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
class ContentStore:
    def __init__(self):
        self._store: dict = {}

    def add(self, user_id: str, prompt: str, content: str) -> str:
        cid = secrets.token_urlsafe(8)
        self._store[cid] = {
            "content_id": cid,
            "user_id":    user_id,
            "prompt":     prompt,
            "content":    content,
            "created_at": datetime.utcnow().isoformat() + "Z",
        }
        return cid

    def get(self, content_id: str) -> dict | None:
        return self._store.get(content_id)

    def list_user(self, user_id: str) -> list[dict]:
        return [v for v in self._store.values() if v["user_id"] == user_id]

    def count(self, user_id: str) -> int:
        return sum(1 for v in self._store.values() if v["user_id"] == user_id)
```

**Why `secrets.token_urlsafe(8)` not `uuid4()`?** Both produce unique IDs. `token_urlsafe` generates URL-safe base64 — shorter (11 chars vs 36) and safe to use directly in URLs. Use `secrets` not `random` — secrets uses the OS CSPRNG, random does not.

</details>